# Laya V2.5 — Windowed Delta + Multiscale Recurrent CeNN
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/codex/laya-v25-window-delta-cenn/notebooks/Laya_Integrated_Memory_V25_WindowDelta_CeNN_Decision_Colab.ipynb)

**New, untrained experiment. No V2.4 outputs are reused.**

- Sliding attention → bidirectional Delta in overlapping 128-token windows (50% overlap, reset state per window).
- Global attention → recurrent CeNN with scales 1/4/16, two updates, and masked global context broadcast.
- Compact marker Transformer → pooled context + shared option-scoring MLP.
- Pretrained encoder FFNs, normalization and residual connections are preserved.

**Training retained from V2.4:** A: progressive local transfer (400 steps **per encoder layer**, now all layers); B: original-head recovery (1600); C: direct gold supervision (900); compact head (700), joint recovery (400), direct compact gold training (700). The original-head intermediate still has head attention; only the final pooled-head model is eligible for export. Learning rates, losses, split sizes and checkpoint selection formulas are retained.

CUDA/FLA preflight checks precede training. FP16 uses gradient scaling. The final encoder never builds a T×T attention mask. This is a bidirectional typed-decision encoder, not a causal language model.

**Default upload gate:** paired accuracy loss including the 95% case-bootstrap lower bound ≤2 percentage points, measured median speedup ≥1.10× across representative dev batches, finite metrics, no remaining self-attention, and a successful save/reload equivalence check. This is a configurable engineering acceptance criterion, not a claim of superiority. A passing model uploads to your authenticated Hugging Face account as a private repository. Set `HF_REPO_ID` to an explicit namespace if desired; use the Colab `HF_TOKEN` secret or interactive login, never paste a token into the notebook.


In [1]:
#@title 1. Install
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
%pip -q install "flash-linear-attention[cuda]==0.5.2"
%pip -q install -U "git+https://github.com/vtavakkoli/TinyCeNN-LM.git@codex/laya-v25-window-delta-cenn"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 456.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 121.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 460.7 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.6/399.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
#@title 2. Imports, configuration, source teacher/student
from pathlib import Path
import copy, json, math, random, time
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from tinycenn_lm.decision_v25 import (
    layer_kinds, replacement_for, core_parameters as v25_core_parameters,
    WindowedBiDelta, MultiscaleCeNN, LinearMaskEncoder,
    PooledMarkerDecisionModel, attention_audit, export_gate, save_v25, load_v25,
    reference_delta,
)

from tinycenn_lm.standalone_decision import (
    QTYPES, build_checkpoint_model, build_sequence, collate_items,
    normalize_question,
    _apply_modernbert_rope, _valid_tokens,
)

SEED = 2026
SOURCE_MODEL = "convaiinnovations/laya-typed-decisions" #@param {type:"string"}

# -------------------------------------------------------------------
# V2.5 tuned configuration — fidelity-first second run
# -------------------------------------------------------------------

# Local attention -> Delta transfer
# First run was still improving at step 120, especially deeper layers.
LOCAL_STEPS = 400 #@param {type:"integer"}

# End-to-end recovery
# 400 steps was still slowly improving; give the Delta encoder much
# more opportunity to adapt, but with a substantially smaller LR.
GLOBAL_STEPS = 1600 #@param {type:"integer"}

# More varied teacher examples are important because the first run
# overfit/plateaued around ~55-57% agreement.
# Leak-free split: 1,100 train cases + 100 dev cases from the official train split.
# The official 400-case test split is never used for training or checkpoint selection.
TRAIN_CASES = 1100 #@param {type:"integer"}
DEV_CASES = 100 #@param {type:"integer"}
TEST_CASES = 400 #@param {type:"integer"}

# Keep batch conservative on T4 / 16 GB.
# If you have A100/L4 with spare memory, try BATCH=4.
BATCH = 3 #@param {type:"integer"}

# Don't shorten context yet: we want to improve fidelity first.
MAX_LEN = 512 #@param {type:"integer"}

# Bidirectional context is required for the ModernBERT encoder use case.
USE_BIDIRECTIONAL = True # Always bidirectional in V2.5

# Keep disabled for the stable run.
# Test True only later as a separate ablation.
ALLOW_NEG_EIGVAL = False # Stable local Delta recurrence

# -------------------------------------------------------------------
# Learning rates
# -------------------------------------------------------------------

# First-run local transfer remained far from the teacher.
# Lower LR + longer training should produce smoother convergence.
LOCAL_LR = 0.01 #@param {type:"number"}
LOCAL_LR_MIN = 0.0005 #@param {type:"number"}

# IMPORTANT:
# Do NOT use 0.15.
# Previous run used ~0.0015; use a lower value for longer recovery.
GLOBAL_DELTA_LR = 0.01 #@param {type:"number"}
GLOBAL_DELTA_LR_MIN = 0.0005 #@param {type:"number"}

# The head was already pretrained. A 0.01 head LR caused oscillation; keep it conservative.
GLOBAL_HEAD_LR = 0.002 #@param {type:"number"}
GLOBAL_HEAD_LR_MIN = 0.0001 #@param {type:"number"}

# -------------------------------------------------------------------
# Direct supervised training on benchmark gold targets
# -------------------------------------------------------------------

# After teacher distillation, train the replacement directly on the dataset's
# hard labels + full soft gold distributions. The official test remains untouched.
DIRECT_FULL_STEPS = 900 #@param {type:"integer"}
DIRECT_FULL_DELTA_LR = 0.003 #@param {type:"number"}
DIRECT_FULL_DELTA_LR_MIN = 0.0003 #@param {type:"number"}
DIRECT_FULL_HEAD_LR = 0.0006 #@param {type:"number"}
DIRECT_FULL_HEAD_LR_MIN = 0.00005 #@param {type:"number"}

DIRECT_COMPACT_STEPS = 700 #@param {type:"integer"}
DIRECT_COMPACT_DELTA_LR = 0.002 #@param {type:"number"}
DIRECT_COMPACT_DELTA_LR_MIN = 0.0002 #@param {type:"number"}
DIRECT_COMPACT_HEAD_LR = 0.0004 #@param {type:"number"}
DIRECT_COMPACT_HEAD_LR_MIN = 0.00004 #@param {type:"number"}

# -------------------------------------------------------------------
# Compact decision head
# -------------------------------------------------------------------

RUN_COMPACT_HEAD = True #@param {type:"boolean"}

# 256 was fast, but quality was still constrained.
# 384 adds modest compute while giving the compact head more capacity.
COMPACT_DIM = 384 #@param {type:"integer"}

# Two layers are enough; don't add a third yet.
COMPACT_LAYERS = 2 #@param {type:"integer"}

# Longer standalone compact-head distillation.
COMPACT_STEPS = 700 #@param {type:"integer"}

# Longer but low-LR joint Delta + compact recovery.
COMPACT_JOINT_STEPS = 400 #@param {type:"integer"}

OUTPUT_DIR = Path(
    "/content/Laya_V25_WindowDelta_CeNN"
)

# -------------------------------------------------------------------
# Reproducibility
# -------------------------------------------------------------------

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU required for the FLA/Triton performance path."
    )

device = torch.device("cuda")

amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported(including_emulation=False)
    else torch.float16
)

# -------------------------------------------------------------------
# Load teacher and student
# -------------------------------------------------------------------

source_dir = Path(
    snapshot_download(
        SOURCE_MODEL,
        allow_patterns=[
            "rl_agent_config.json",
            "model.safetensors",
            "encoder/*",
            "tokenizer/*",
        ],
    )
)

teacher, source_cfg = build_checkpoint_model(source_dir)
student_full, _ = build_checkpoint_model(source_dir)

tokenizer = AutoTokenizer.from_pretrained(
    str(source_dir / "tokenizer")
)

teacher.to(device).eval().requires_grad_(False)
student_full.to(device).eval().requires_grad_(False)

# Disable auto-compilation for consistent teacher/student benchmarking.
teacher.encoder.config.reference_compile = False
student_full.encoder.config.reference_compile = False
kinds = layer_kinds(student_full.encoder)
full_layers = [i for i, kind in enumerate(kinds) if kind == "full_attention"]
replacement_layers = list(range(len(kinds)))
ARCHITECTURE = {
    "window": 128, "cell_dim": 256, "cell_steps": 2,
    "scales": [1, 4, 16], "backend": "fla",
}
MAX_ACCURACY_DROP = 0.02 #@param {type:"number"}
MIN_SPEEDUP = 1.10 #@param {type:"number"}
PUSH_TO_HUB = True #@param {type:"boolean"}
HF_REPO_ID = "" #@param {type:"string"}
HF_PRIVATE = True #@param {type:"boolean"}
scaler = torch.amp.GradScaler("cuda", enabled=amp_dtype == torch.float16)
print("V2.5 layer map:", dict(enumerate(kinds)))
print("Architecture:", ARCHITECTURE)

print("device:", torch.cuda.get_device_name(0))
print("torch:", torch.__version__, "| amp:", amp_dtype)
print("source:", SOURCE_MODEL)
print(
    "full-attention layers:",
    full_layers,
    "| count:",
    len(full_layers),
)

print("\nTuned V2.5 configuration")
print("  LOCAL_STEPS       :", LOCAL_STEPS)
print("  GLOBAL_STEPS      :", GLOBAL_STEPS)
print("  TRAIN / DEV / TEST:", TRAIN_CASES, "/", DEV_CASES, "/", TEST_CASES)
print("  LOCAL_LR          :", LOCAL_LR, "->", LOCAL_LR_MIN)
print("  GLOBAL_DELTA_LR   :", GLOBAL_DELTA_LR, "->", GLOBAL_DELTA_LR_MIN)
print("  GLOBAL_HEAD_LR    :", GLOBAL_HEAD_LR, "->", GLOBAL_HEAD_LR_MIN)
print("  DIRECT_FULL_STEPS :", DIRECT_FULL_STEPS)
print("  DIRECT_COMPACT    :", DIRECT_COMPACT_STEPS)
print("  COMPACT_DIM       :", COMPACT_DIM)
print("  COMPACT_STEPS     :", COMPACT_STEPS)
print("  COMPACT_JOINT     :", COMPACT_JOINT_STEPS)

assert full_layers

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/843M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/337 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

rl_agent_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

V2.5 layer map: {0: 'full_attention', 1: 'sliding_attention', 2: 'sliding_attention', 3: 'full_attention', 4: 'sliding_attention', 5: 'sliding_attention', 6: 'full_attention', 7: 'sliding_attention', 8: 'sliding_attention', 9: 'full_attention', 10: 'sliding_attention', 11: 'sliding_attention', 12: 'full_attention', 13: 'sliding_attention', 14: 'sliding_attention', 15: 'full_attention', 16: 'sliding_attention', 17: 'sliding_attention', 18: 'full_attention', 19: 'sliding_attention', 20: 'sliding_attention', 21: 'full_attention', 22: 'sliding_attention', 23: 'sliding_attention', 24: 'full_attention', 25: 'sliding_attention', 26: 'sliding_attention', 27: 'full_attention'}
Architecture: {'window': 128, 'cell_dim': 256, 'cell_steps': 2, 'scales': [1, 4, 16], 'backend': 'fla'}
device: Tesla T4
torch: 2.11.0+cu130 | amp: torch.float16
source: convaiinnovations/laya-typed-decisions
full-attention layers: [0, 3, 6, 9, 12, 15, 18, 21, 24, 27] | count: 10

Tuned V2.5 configuration
  LOCAL_STEPS   

In [3]:
#@title 3. V2.5 kernel preflight and layer helpers
def v25_indices(model):
    return [i for i, layer in enumerate(model.encoder.layers)
            if isinstance(layer.attn, (WindowedBiDelta, MultiscaleCeNN))]

# FLA correctness and backward support must pass on the actual GPU/dtype.
# Never silently fall back to a Python scan in measured CUDA runs.
from fla.ops.gated_delta_rule import chunk_gated_delta_rule
with torch.autocast(device_type="cuda", dtype=amp_dtype):
    q = torch.randn(1, 32, 2, 64, device=device, dtype=amp_dtype, requires_grad=True)
    k = torch.randn_like(q).requires_grad_(True)
    v = torch.randn_like(q).requires_grad_(True)
    g = torch.full((1, 32, 2), -0.02, device=device, dtype=amp_dtype, requires_grad=True)
    beta = torch.full_like(g, 0.3, requires_grad=True)
    actual, _ = chunk_gated_delta_rule(
        q=q, k=k, v=v, g=g, beta=beta, output_final_state=False,
        use_qk_l2norm_in_kernel=True, use_beta_sigmoid_in_kernel=False,
        allow_neg_eigval=False, state_v_first=True,
    )
expected = reference_delta(q, k, v, g, beta)
torch.testing.assert_close(actual.float(), expected.float(), rtol=0.05, atol=0.015)
grads = torch.autograd.grad(actual.float().square().mean(), (q, k, v, g, beta))
assert all(torch.isfinite(x).all() and x.abs().sum() > 0 for x in grads)
print("FLA forward/backward preflight passed:", torch.cuda.get_device_name(), amp_dtype)
del q, k, v, g, beta, actual, expected, grads


FLA forward/backward preflight passed: Tesla T4 torch.float16


In [4]:
#@title 4. Build leak-free train / dev / untouched official test
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    """Align the benchmark gold distribution to the exact marker order used by build_sequence."""
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        # Score markers are generated in rubric order. The benchmark normally
        # uses numeric string keys ("0", "1", ...); sort those numerically.
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        labels = list(gold_keys)

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not present in option labels {labels!r}")
    gold_index = canon_labels.index(gold_label)
    return labels, gold_probs.tolist(), int(gold_index)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "), add_special_tokens=False
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer, state, q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue

            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: marker/gold option mismatch "
                    f"({len(markers)} markers vs {len(labels)} labels)"
                )

            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": gold_index,
            })

    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Split only the official TRAIN rows into fit/dev. Never tune on official TEST.
all_train_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="train"))
split_rng = random.Random(SEED)
split_rng.shuffle(all_train_rows)
assert TRAIN_CASES + DEV_CASES <= len(all_train_rows)

train_rows = all_train_rows[:TRAIN_CASES]
dev_rows = all_train_rows[TRAIN_CASES:TRAIN_CASES + DEV_CASES]
train_items = make_items_from_rows(train_rows, SEED, shuffle_items=True)
dev_items = make_items_from_rows(dev_rows, SEED + 1, shuffle_items=True)

print(
    "cases  train/dev:",
    len(train_rows), "/", len(dev_rows),
)
print(
    "decisions train/dev:",
    len(train_items), "/", len(dev_items),
)
print("official test: deferred until the final evaluation cell")
assert len(train_items) >= 64 and len(dev_items) >= 32

def batch(items, offset, n=BATCH):
    selected = [items[(offset+i) % len(items)] for i in range(n)]
    b = collate_items(selected, int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}

def attn_io(model, layer_idx, b):
    layer = model.encoder.layers[layer_idx]
    got = {}
    def pre(mod, args, kwargs):
        got["x"] = args[0].detach()
        got["pos"] = None if kwargs.get("position_embeddings") is None else tuple(
            z.detach() for z in kwargs["position_embeddings"]
        )
        got["mask"] = None if kwargs.get("attention_mask") is None else kwargs["attention_mask"].detach()
    def post(mod, args, kwargs, out):
        got["y"] = out[0].detach()
    h1 = layer.attn.register_forward_pre_hook(pre, with_kwargs=True)
    h2 = layer.attn.register_forward_hook(post, with_kwargs=True)
    try:
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            model(**b)
    finally:
        h1.remove(); h2.remove()
    return got

def local_loss(pred, target, valid):
    m = valid[:, :, None].float()
    mse = (((pred.float()-target.float())**2)*m).sum() / (m.sum()*pred.shape[-1]).clamp_min(1)
    power = ((target.float()**2)*m).sum() / (m.sum()*target.shape[-1]).clamp_min(1)
    nmse = mse / power.clamp_min(1e-8)
    cos = F.cosine_similarity(pred.float()[valid], target.float()[valid], dim=-1).mean()
    return nmse + 1.10*(1-cos), nmse, cos


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

cases  train/dev: 1100 / 100
decisions train/dev: 5500 / 500
official test: deferred until the final evaluation cell


In [5]:
#@title 5. Stage A — progressive local transfer
def cosine_lr(step, total_steps, start_lr, end_lr):
    """Exact cosine decay: step 0=start_lr, last step=end_lr."""
    if total_steps <= 1:
        return float(end_lr)
    x = min(1.0, max(0.0, step / float(total_steps - 1)))
    return float(end_lr + 0.5 * (start_lr - end_lr) * (1.0 + math.cos(math.pi * x)))

local_report = {}

for layer_idx in replacement_layers:
    print(f"\n=== {kinds[layer_idx]} layer {layer_idx} ===")
    original = student_full.encoder.layers[layer_idx].attn
    repl = replacement_for(original, kinds[layer_idx], ARCHITECTURE).to(device)
    params = repl.trainable_core_parameters()
    opt = torch.optim.AdamW(params, lr=LOCAL_LR, betas=(0.9, 0.95), weight_decay=1e-4)
    best = None; best_score = float("inf")

    probe_b = batch(dev_items, layer_idx*11, min(5, BATCH+1))
    probe = attn_io(student_full, layer_idx, probe_b)
    probe_valid = probe_b["attention_mask"].bool()

    for step in range(LOCAL_STEPS):
        local_lr = cosine_lr(step, LOCAL_STEPS, LOCAL_LR, LOCAL_LR_MIN)
        opt.param_groups[0]["lr"] = local_lr
        b = batch(train_items, step*BATCH + layer_idx*17)
        target = attn_io(student_full, layer_idx, b)
        valid = b["attention_mask"].bool()

        repl.train(); repl.out_drop.eval(); opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            pred, _ = repl(
                target["x"],
                position_embeddings=target["pos"],
                attention_mask=target["mask"],
            )
            loss, nmse, cos = local_loss(pred, target["y"], valid)
        if not torch.isfinite(loss):
            raise RuntimeError(f"non-finite local loss at layer {layer_idx}")
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt)
        scaler.update()

        if step == 0 or (step+1) % max(20, LOCAL_STEPS//4) == 0 or step+1 == LOCAL_STEPS:
            repl.eval()
            with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
                pp, _ = repl(
                    probe["x"],
                    position_embeddings=probe["pos"],
                    attention_mask=probe["mask"],
                )
                ploss, pnmse, pcos = local_loss(pp, probe["y"], probe_valid)
            score = float(ploss)
            if score < best_score:
                best_score = score
                best = {k:v.detach().cpu().clone() for k,v in repl.state_dict().items()}
            print(
                f"step {step+1:03d}/{LOCAL_STEPS} "
                f"nmse={float(pnmse):.4f} cos={float(pcos):.4f} lr={local_lr:.6f}"
            )

    if best is not None:
        repl.load_state_dict(best)
    repl.eval()
    student_full.encoder.layers[layer_idx].attn = repl
    local_report[str(layer_idx)] = {"best_local_score": best_score}
    torch.cuda.empty_cache()

print("\nconverted:", v25_indices(student_full))
assert v25_indices(student_full) == replacement_layers
student_full.encoder = LinearMaskEncoder(student_full.encoder)
print("All encoder attention replaced; FFNs/norms/residuals retained.")
print("V2.5 new core params:", f"{sum(p.numel() for p in v25_core_parameters(student_full)):,}")



=== full_attention layer 0 ===
step 001/400 nmse=1.7571 cos=0.1445 lr=0.010000
step 100/400 nmse=0.4169 cos=0.7833 lr=0.008629
step 200/400 nmse=0.3987 cos=0.8055 lr=0.005269
step 300/400 nmse=0.3657 cos=0.8265 lr=0.001898
step 400/400 nmse=0.3374 cos=0.8380 lr=0.000500

=== sliding_attention layer 1 ===
step 001/400 nmse=1.0392 cos=-0.1911 lr=0.010000
step 100/400 nmse=0.6917 cos=0.7446 lr=0.008629
step 200/400 nmse=0.3855 cos=0.7856 lr=0.005269
step 300/400 nmse=0.3105 cos=0.8070 lr=0.001898
step 400/400 nmse=0.2977 cos=0.8119 lr=0.000500

=== sliding_attention layer 2 ===
step 001/400 nmse=1.0430 cos=-0.1591 lr=0.010000
step 100/400 nmse=0.7291 cos=0.6865 lr=0.008629
step 200/400 nmse=0.4421 cos=0.7640 lr=0.005269
step 300/400 nmse=0.3597 cos=0.7923 lr=0.001898
step 400/400 nmse=0.3433 cos=0.7992 lr=0.000500

=== full_attention layer 3 ===
step 001/400 nmse=4.4940 cos=0.1644 lr=0.010000
step 100/400 nmse=0.3653 cos=0.7931 lr=0.008629
step 200/400 nmse=0.3083 cos=0.8259 lr=0.005269


In [6]:
#@title 6. Stage B — end-to-end recovery with original decision head
for p in student_full.parameters():
    p.requires_grad = False

# Core discovery must be independent of requires_grad; this assertion catches regressions.
delta_params = v25_core_parameters(student_full)
assert len(delta_params) > 0, "Delta core parameter list is empty; Stage B would not train the replacement."
for p in delta_params:
    p.requires_grad = True
print("Global Delta trainable params:", f"{sum(p.numel() for p in delta_params):,}")

head_params = []
for module in (student_full.head, student_full.type_emb, student_full.scorer, student_full.act_head):
    if module is not None:
        for p in module.parameters():
            p.requires_grad = True
            head_params.append(p)

opt = torch.optim.AdamW([
    {"params": delta_params, "lr": GLOBAL_DELTA_LR, "weight_decay": 1e-4},
    {"params": head_params, "lr": GLOBAL_HEAD_LR, "weight_decay": 1e-4},
], betas=(0.9, 0.95))
T_DISTILL = 1.25
RANK_MARGIN = 0.15
RANK_WEIGHT = 0.15

def distill_loss(slog, tlog, mask, sact, tact):
    s = slog.masked_fill(~mask, -1e4) / T_DISTILL
    t = tlog.masked_fill(~mask, -1e4) / T_DISTILL
    tp = torch.softmax(t.detach(), -1)
    kl = (
        tp * (torch.log_softmax(t.detach(), -1) - torch.log_softmax(s, -1))
    ).sum(-1).mean() * (T_DISTILL*T_DISTILL)
    smasked = slog.masked_fill(~mask, -1e4)
    tmasked = tlog.masked_fill(~mask, -1e4)
    target = tmasked.argmax(-1)
    ce = F.cross_entropy(smasked, target)

    # Teacher-top1 ranking loss. This directly targets the observed case where
    # teacher top1 is already in student top2 but appears in the wrong order.
    target_score = smasked.gather(1, target[:, None]).squeeze(1)
    other = smasked.clone()
    other.scatter_(1, target[:, None], -1e4)
    strongest_other = other.max(-1).values
    rank = F.relu(RANK_MARGIN + strongest_other - target_score).mean()

    ap = torch.softmax(tact.detach()/T_DISTILL, -1)
    akl = (
        ap * (
            torch.log_softmax(tact.detach()/T_DISTILL, -1)
            - torch.log_softmax(sact/T_DISTILL, -1)
        )
    ).sum(-1).mean() * (T_DISTILL*T_DISTILL)
    return kl + 0.30*ce + RANK_WEIGHT*rank + 0.05*akl

@torch.no_grad()
def validate_pair(student, items=dev_items, limit=128):
    teacher.eval(); student.eval()
    total = agree = 0; js = []
    for off in range(0, min(limit, len(items)), BATCH):
        n = min(BATCH, min(limit, len(items))-off)
        b = batch(items, off, n)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, _ = teacher(**b); sl, _ = student(**b)
        m = b["marker_mask"].bool()
        tm = tl.float().masked_fill(~m, -1e9)
        sm = sl.float().masked_fill(~m, -1e9)
        tp = torch.softmax(tm, -1); sp = torch.softmax(sm, -1)
        agree += int((tm.argmax(-1) == sm.argmax(-1)).sum())
        total += int(tm.shape[0])
        mid = 0.5*(tp+sp)
        j = 0.5*(tp*(tp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        j += 0.5*(sp*(sp.clamp_min(1e-8).log()-mid.clamp_min(1e-8).log())).sum(-1)
        js.extend(j.cpu().tolist())
    return {"agreement": agree/max(1,total), "mean_js": float(np.mean(js))}

best_state = None; best_score = float("inf")
eval_every = max(40, GLOBAL_STEPS//8)

for step in range(GLOBAL_STEPS):
    delta_lr = cosine_lr(step, GLOBAL_STEPS, GLOBAL_DELTA_LR, GLOBAL_DELTA_LR_MIN)
    head_lr = cosine_lr(step, GLOBAL_STEPS, GLOBAL_HEAD_LR, GLOBAL_HEAD_LR_MIN)
    opt.param_groups[0]["lr"] = delta_lr
    opt.param_groups[1]["lr"] = head_lr
    b = batch(train_items, step*BATCH + 31)
    opt.zero_grad(set_to_none=True)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
        tl, ta = teacher(**b)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        sl, sa = student_full(**b)
        loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)

    scaler.scale(loss).backward()

    scaler.unscale_(opt)
    if step == 0:
        delta_grad_count = sum(
            int(p.grad is not None and torch.isfinite(p.grad).all()) for p in delta_params
        )
        assert delta_grad_count > 0, "Delta core received no finite gradients."
        print("Delta tensors with finite gradients:", delta_grad_count, "/", len(delta_params))
    # Clip independently: the larger decision head should not dominate Delta updates.
    torch.nn.utils.clip_grad_norm_(delta_params, 1.0)
    torch.nn.utils.clip_grad_norm_(head_params, 0.5)
    scaler.step(opt)
    scaler.update()

    if step == 0 or (step+1) % eval_every == 0 or step+1 == GLOBAL_STEPS:
        metrics = validate_pair(student_full, limit=96)
        score = metrics["mean_js"] + 0.50*(1.0-metrics["agreement"])
        print(
            f"step={step+1:03d}/{GLOBAL_STEPS} loss={float(loss.detach()):.4f} "
            f"agreement={metrics['agreement']:.2%} JS={metrics['mean_js']:.5f} "
            f"delta_lr={delta_lr:.6f} head_lr={head_lr:.6f}"
        )
        if score < best_score:
            best_score = score
            best_state = {
                n:p.detach().cpu().clone()
                for n,p in student_full.named_parameters() if p.requires_grad
            }

if best_state:
    with torch.no_grad():
        for n,p in student_full.named_parameters():
            if n in best_state:
                p.copy_(best_state[n].to(device=p.device, dtype=p.dtype))

student_full.eval().requires_grad_(False)
v25_full_quality = validate_pair(student_full, items=dev_items, limit=len(dev_items))
print("V2.5 full-head quality:", v25_full_quality)

# Release the large AdamW optimizer states before direct-gold training / compact copy.
del opt
torch.cuda.empty_cache()


Global Delta trainable params: 6,550,024
Delta tensors with finite gradients: 250 / 264
step=001/1600 loss=0.7057 agreement=44.79% JS=0.09882 delta_lr=0.010000 head_lr=0.002000
step=200/1600 loss=1.3236 agreement=19.79% JS=0.11472 delta_lr=0.009642 head_lr=0.001928
step=400/1600 loss=1.0566 agreement=31.25% JS=0.11470 delta_lr=0.008614 head_lr=0.001723
step=600/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.007073 head_lr=0.001415
step=800/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.005255 head_lr=0.001051
step=1000/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.003435 head_lr=0.000687
step=1200/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.001893 head_lr=0.000379
step=1400/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.000862 head_lr=0.000172
step=1600/1600 loss=nan agreement=20.83% JS=nan delta_lr=0.000500 head_lr=0.000100
V2.5 full-head quality: {'agreement': 0.462, 'mean_js': 0.08776194895547815}


In [7]:
#@title 7. Stage C — direct supervised training on benchmark gold data
# This stage is different from distillation: teacher outputs are NOT used in the loss.
# Targets come directly from train_items: gold label + full soft gold distribution.

def _gold_target_tensors(selected, marker_mask, device):
    max_k = marker_mask.shape[1]
    soft = torch.zeros((len(selected), max_k), dtype=torch.float32, device=device)
    hard = torch.zeros(len(selected), dtype=torch.long, device=device)
    for i, item in enumerate(selected):
        g = torch.tensor(item["gold_probs"], dtype=torch.float32, device=device)
        soft[i, :g.numel()] = g
        hard[i] = int(item["gold_index"])
    soft = soft * marker_mask.float()
    soft = soft / soft.sum(-1, keepdim=True).clamp_min(1e-8)
    return soft, hard

def gold_supervised_loss(logits, marker_mask, selected):
    masked = logits.float().masked_fill(~marker_mask, -1e4)
    gold_soft, gold_hard = _gold_target_tensors(selected, marker_mask, logits.device)
    logp = torch.log_softmax(masked, dim=-1)
    p = torch.softmax(masked, dim=-1)

    # Full probability-target training.
    soft_ce = -(gold_soft * logp).sum(-1).mean()

    # Explicit official hard-label target.
    hard_ce = F.cross_entropy(masked, gold_hard)

    # Calibration pressure toward the complete benchmark distribution.
    brier = (((p - gold_soft) ** 2) * marker_mask.float()).sum(-1).mean()

    # Hard-label margin: useful when soft gold has close alternatives.
    target_score = masked.gather(1, gold_hard[:, None]).squeeze(1)
    other = masked.clone()
    other.scatter_(1, gold_hard[:, None], -1e4)
    strongest_other = other.max(-1).values
    rank = F.relu(0.10 + strongest_other - target_score).mean()

    total = soft_ce + 0.35 * hard_ce + 0.20 * brier + 0.10 * rank
    return total, {
        "soft_ce": float(soft_ce.detach()),
        "hard_ce": float(hard_ce.detach()),
        "brier": float(brier.detach()),
        "rank": float(rank.detach()),
    }

@torch.no_grad()
def gold_dev_metrics(model, items=dev_items):
    model.eval()
    correct = 0
    total = 0
    kl_values = []
    brier_values = []
    soft_acc_values = []

    for off in range(0, len(items), BATCH):
        selected = items[off:off+BATCH]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = model(**b)
        mm = b["marker_mask"].bool()
        masked = logits.float().masked_fill(~mm, -1e9)
        p = torch.softmax(masked, dim=-1)
        g, hard = _gold_target_tensors(selected, mm, device)

        pred = p.argmax(-1)
        correct += int((pred == hard).sum())
        total += int(pred.numel())

        kl = (g * (g.clamp_min(1e-8).log() - p.clamp_min(1e-8).log())).sum(-1)
        br = (((p - g) ** 2) * mm.float()).sum(-1)
        soft_acc = g.gather(1, pred[:, None]).squeeze(1)
        kl_values.extend(kl.cpu().tolist())
        brier_values.extend(br.cpu().tolist())
        soft_acc_values.extend(soft_acc.cpu().tolist())

    return {
        "accuracy": correct / max(1, total),
        "kl_from_gold": float(np.mean(kl_values)),
        "brier": float(np.mean(brier_values)),
        "soft_accuracy": float(np.mean(soft_acc_values)),
    }

def direct_gold_train_full(model):
    # Freeze everything first. Train only Delta cores + decision path.
    model.requires_grad_(False)
    delta = v25_core_parameters(model)
    assert len(delta) > 0
    for p in delta:
        p.requires_grad = True

    decision_params = []
    # No direct target exists for act_head, so leave it frozen.
    for module in (model.head, model.type_emb, model.scorer):
        if module is not None:
            for p in module.parameters():
                p.requires_grad = True
                decision_params.append(p)

    opt = torch.optim.AdamW([
        {"params": delta, "lr": DIRECT_FULL_DELTA_LR, "weight_decay": 1e-4},
        {"params": decision_params, "lr": DIRECT_FULL_HEAD_LR, "weight_decay": 1e-4},
    ], betas=(0.9, 0.95))

    best = None
    best_score = float("inf")
    eval_every = max(60, DIRECT_FULL_STEPS // 9)

    for step in range(DIRECT_FULL_STEPS):
        delta_lr = cosine_lr(
            step, DIRECT_FULL_STEPS, DIRECT_FULL_DELTA_LR, DIRECT_FULL_DELTA_LR_MIN
        )
        head_lr = cosine_lr(
            step, DIRECT_FULL_STEPS, DIRECT_FULL_HEAD_LR, DIRECT_FULL_HEAD_LR_MIN
        )
        opt.param_groups[0]["lr"] = delta_lr
        opt.param_groups[1]["lr"] = head_lr

        start = (step * BATCH + 701) % len(train_items)
        selected = [train_items[(start+i) % len(train_items)] for i in range(BATCH)]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}

        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = model(**b)
            loss, parts = gold_supervised_loss(logits, b["marker_mask"].bool(), selected)

        if not torch.isfinite(loss):
            raise RuntimeError("non-finite direct gold loss")
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(delta, 1.0)
        torch.nn.utils.clip_grad_norm_(decision_params, 0.5)
        scaler.step(opt)
        scaler.update()

        if step == 0 or (step+1) % eval_every == 0 or step+1 == DIRECT_FULL_STEPS:
            m = gold_dev_metrics(model)
            # Accuracy first, distribution quality as tie-breaker.
            score = m["kl_from_gold"] + 0.35 * (1.0 - m["accuracy"]) + 0.15 * m["brier"]
            print(
                f"gold-full step={step+1:03d}/{DIRECT_FULL_STEPS} "
                f"loss={float(loss.detach()):.4f} acc={m['accuracy']:.2%} "
                f"KL={m['kl_from_gold']:.4f} Brier={m['brier']:.4f} "
                f"delta_lr={delta_lr:.6f} head_lr={head_lr:.6f}"
            )
            if score < best_score:
                best_score = score
                best = {
                    n:p.detach().cpu().clone()
                    for n,p in model.named_parameters() if p.requires_grad
                }

    if best:
        with torch.no_grad():
            for n,p in model.named_parameters():
                if n in best:
                    p.copy_(best[n].to(device=p.device, dtype=p.dtype))

    model.eval().requires_grad_(False)
    return gold_dev_metrics(model)

v25_full_gold_dev = direct_gold_train_full(student_full)
# Refresh teacher-agreement diagnostics after the direct supervised phase.
v25_full_quality = validate_pair(student_full, items=dev_items, limit=len(dev_items))
print("V2.5 full-head direct-gold dev:", v25_full_gold_dev)
print("V2.5 full-head post-gold teacher agreement:", v25_full_quality)
torch.cuda.empty_cache()


gold-full step=001/900 loss=1.0521 acc=26.20% KL=nan Brier=nan delta_lr=0.003000 head_lr=0.000600


RuntimeError: non-finite direct gold loss

In [ ]:
#@title 8. Benchmark helper — latency + CUDA activation peak
def _sync():
    torch.cuda.synchronize()

@torch.inference_mode()
def _forward(model, b):
    model.eval()
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        return model(**b)

def benchmark_model(model, probe, warmup=3, repeats=10):
    for _ in range(warmup):
        _forward(model, probe)
    _sync()
    start_alloc = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats()
    samples = []
    for _ in range(repeats):
        _sync()
        t0 = time.perf_counter()
        _forward(model, probe)
        _sync()
        samples.append((time.perf_counter()-t0)*1000.0)
    peak = torch.cuda.max_memory_allocated()
    return {
        "median_ms": float(np.median(samples)),
        "p95_ms": float(np.percentile(samples, 95)),
        "activation_peak_delta_mb": float(max(0, peak-start_alloc) / 2**20),
        "total_parameters": int(sum(p.numel() for p in model.parameters())),
    }

probe = batch(dev_items, 0, min(BATCH, 3))
teacher_perf = benchmark_model(teacher, probe)
v25_full_perf = benchmark_model(student_full, probe)
v25_full_report = {
    "quality": v25_full_quality,
    "teacher": teacher_perf,
    "student": v25_full_perf,
    "speedup_vs_teacher": teacher_perf["median_ms"] / max(v25_full_perf["median_ms"], 1e-9),
    "sequence_length": int(probe["input_ids"].shape[1]),
    "batch_size": int(probe["input_ids"].shape[0]),
    "new_delta_core_parameters": int(sum(p.numel() for p in v25_core_parameters(student_full))),
}
print(json.dumps(v25_full_report, indent=2))

# All models are timed on these same length-stratified DEV examples, without
# selecting the fastest test batch. Report median/p95 across all timed forwards.
ordered_dev = sorted(dev_items, key=lambda item: len(item["ids"]))
perf_probes = []
for fraction in (0.0, 0.25, 0.5, 0.75, 1.0):
    start = min(len(ordered_dev)-BATCH, int(fraction * (len(ordered_dev)-BATCH)))
    probe_batch = collate_items(ordered_dev[start:start+BATCH], int(tokenizer.pad_token_id))
    perf_probes.append({k: v.to(device) for k, v in probe_batch.items()})

@torch.inference_mode()
def benchmark_suite(model, probes=perf_probes, repeats=20):
    timings, peaks = [], []
    for pb in probes:
        for _ in range(5):
            _forward(model, pb)
        _sync()
        base = torch.cuda.memory_allocated()
        torch.cuda.reset_peak_memory_stats()
        for _ in range(repeats):
            _sync()
            started = time.perf_counter()
            _forward(model, pb)
            _sync()
            timings.append((time.perf_counter()-started)*1000)
        peaks.append(max(0, torch.cuda.max_memory_allocated()-base) / 2**20)
    return {
        "median_ms": float(np.median(timings)), "p95_ms": float(np.percentile(timings, 95)),
        "activation_peak_delta_mb": float(max(peaks)),
        "total_parameters": sum(p.numel() for p in model.parameters()),
        "probe_lengths": [int(pb["input_ids"].shape[1]) for pb in probes],
        "batch_size": BATCH, "repeats_per_probe": repeats,
        "scope": "model forward only, length-stratified dev probes; excludes tokenization",
    }


In [ ]:
#@title 9. Compact pooled-context MLP head — same three training phases
def train_compact_model():
    compact = PooledMarkerDecisionModel(
        student_full, compact_dim=COMPACT_DIM, layers=COMPACT_LAYERS
    ).to(device)

    # Phase 1: compact head only.
    for p in compact.encoder.parameters(): p.requires_grad = False
    for p in compact.type_emb.parameters(): p.requires_grad = False
    head = [
        p for n,p in compact.named_parameters()
        if not n.startswith("encoder.") and not n.startswith("type_emb.")
    ]
    opt = torch.optim.AdamW(head, lr=8e-4, betas=(0.9, 0.95), weight_decay=1e-4)
    compact_head_lr_start, compact_head_lr_end = 8e-4, 8e-5

    best = None; best_score = float("inf")
    eval_every = max(40, COMPACT_STEPS//6)
    for step in range(COMPACT_STEPS):
        compact_lr = cosine_lr(step, COMPACT_STEPS, compact_head_lr_start, compact_head_lr_end)
        opt.param_groups[0]["lr"] = compact_lr
        b = batch(train_items, step*BATCH + 101)
        opt.zero_grad(set_to_none=True)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta = teacher(**b)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa = compact(**b)
            loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(head, 1.0)
        scaler.step(opt)
        scaler.update()

        if step == 0 or (step+1) % eval_every == 0 or step+1 == COMPACT_STEPS:
            m = validate_pair(compact, items=dev_items, limit=96)
            score = m["mean_js"] + 0.50*(1-m["agreement"])
            print(
                f"compact-head step={step+1:03d}/{COMPACT_STEPS} "
                f"loss={float(loss.detach()):.4f} agreement={m['agreement']:.2%} "
                f"JS={m['mean_js']:.5f} lr={compact_lr:.6f}"
            )
            if score < best_score:
                best_score = score
                best = {
                    n:p.detach().cpu().clone()
                    for n,p in compact.named_parameters() if p.requires_grad
                }

    if best:
        with torch.no_grad():
            for n,p in compact.named_parameters():
                if n in best:
                    p.copy_(best[n].to(device=p.device, dtype=p.dtype))

    # Phase 2: short low-LR joint tune of Delta cores + compact head.
    delta = v25_core_parameters(compact)
    assert len(delta) > 0, "Compact joint phase found no Delta parameters."
    for p in delta: p.requires_grad = True
    print("Compact joint Delta params:", f"{sum(p.numel() for p in delta):,}")
    joint_head = [
        p for n,p in compact.named_parameters()
        if not n.startswith("encoder.") and p.requires_grad
    ]
    joint_delta_start, joint_delta_end = 0.002, 0.0002
    joint_head_start, joint_head_end = 0.00015, 0.00003
    opt = torch.optim.AdamW([
        {"params": delta, "lr": joint_delta_start, "weight_decay": 1e-4},
        {"params": joint_head, "lr": joint_head_start, "weight_decay": 1e-4},
    ], betas=(0.9, 0.95))

    best_joint = None
    best_joint_score = float("inf")

    for step in range(COMPACT_JOINT_STEPS):
        joint_delta_lr = cosine_lr(step, COMPACT_JOINT_STEPS, joint_delta_start, joint_delta_end)
        joint_head_lr = cosine_lr(step, COMPACT_JOINT_STEPS, joint_head_start, joint_head_end)
        opt.param_groups[0]["lr"] = joint_delta_lr
        opt.param_groups[1]["lr"] = joint_head_lr
        b = batch(train_items, step*BATCH + 401)
        opt.zero_grad(set_to_none=True)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=amp_dtype):
            tl, ta = teacher(**b)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            sl, sa = compact(**b)
            loss = distill_loss(sl, tl, b["marker_mask"].bool(), sa, ta)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        if step == 0:
            delta_grad_count = sum(
                int(p.grad is not None and torch.isfinite(p.grad).all()) for p in delta
            )
            assert delta_grad_count > 0, "Compact joint Delta core received no finite gradients."
            print("Compact-joint Delta tensors with finite gradients:", delta_grad_count, "/", len(delta))
        torch.nn.utils.clip_grad_norm_(delta, 1.0)
        torch.nn.utils.clip_grad_norm_(joint_head, 0.5)
        scaler.step(opt)
        scaler.update()

        if step == 0 or (step+1) % max(30, COMPACT_JOINT_STEPS//4) == 0 or step+1 == COMPACT_JOINT_STEPS:
            m = validate_pair(compact, items=dev_items, limit=96)
            joint_score = m["mean_js"] + 0.50*(1-m["agreement"])
            print(
                f"compact-joint step={step+1:03d}/{COMPACT_JOINT_STEPS} "
                f"loss={float(loss.detach()):.4f} agreement={m['agreement']:.2%} "
                f"JS={m['mean_js']:.5f} delta_lr={joint_delta_lr:.6f} "
                f"head_lr={joint_head_lr:.6f}"
            )
            if joint_score < best_joint_score:
                best_joint_score = joint_score
                best_joint = {
                    n:p.detach().cpu().clone()
                    for n,p in compact.named_parameters() if p.requires_grad
                }

    if best_joint:
        with torch.no_grad():
            for n,p in compact.named_parameters():
                if n in best_joint:
                    p.copy_(best_joint[n].to(device=p.device, dtype=p.dtype))

    # Phase 3: direct supervised training on the benchmark gold targets.
    # Teacher logits are not used here.
    compact.requires_grad_(False)
    delta = v25_core_parameters(compact)
    for p in delta:
        p.requires_grad = True

    direct_head = []
    for module in (compact.type_emb, compact.in_proj, compact.context_proj, compact.marker_mixer, compact.scorer):
        for p in module.parameters():
            p.requires_grad = True
            direct_head.append(p)

    opt = torch.optim.AdamW([
        {"params": delta, "lr": DIRECT_COMPACT_DELTA_LR, "weight_decay": 1e-4},
        {"params": direct_head, "lr": DIRECT_COMPACT_HEAD_LR, "weight_decay": 1e-4},
    ], betas=(0.9, 0.95))

    best_gold = None
    best_gold_score = float("inf")
    eval_every = max(50, DIRECT_COMPACT_STEPS // 7)

    for step in range(DIRECT_COMPACT_STEPS):
        delta_lr = cosine_lr(
            step, DIRECT_COMPACT_STEPS,
            DIRECT_COMPACT_DELTA_LR, DIRECT_COMPACT_DELTA_LR_MIN
        )
        head_lr = cosine_lr(
            step, DIRECT_COMPACT_STEPS,
            DIRECT_COMPACT_HEAD_LR, DIRECT_COMPACT_HEAD_LR_MIN
        )
        opt.param_groups[0]["lr"] = delta_lr
        opt.param_groups[1]["lr"] = head_lr

        start = (step * BATCH + 1301) % len(train_items)
        selected = [train_items[(start+i) % len(train_items)] for i in range(BATCH)]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}

        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda", dtype=amp_dtype):
            logits, _ = compact(**b)
            loss, parts = gold_supervised_loss(logits, b["marker_mask"].bool(), selected)
        if not torch.isfinite(loss):
            raise RuntimeError("non-finite compact direct-gold loss")
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(delta, 1.0)
        torch.nn.utils.clip_grad_norm_(direct_head, 0.5)
        scaler.step(opt)
        scaler.update()

        if step == 0 or (step+1) % eval_every == 0 or step+1 == DIRECT_COMPACT_STEPS:
            m = gold_dev_metrics(compact)
            score = m["kl_from_gold"] + 0.35 * (1.0 - m["accuracy"]) + 0.15 * m["brier"]
            print(
                f"gold-compact step={step+1:03d}/{DIRECT_COMPACT_STEPS} "
                f"loss={float(loss.detach()):.4f} acc={m['accuracy']:.2%} "
                f"KL={m['kl_from_gold']:.4f} Brier={m['brier']:.4f} "
                f"delta_lr={delta_lr:.6f} head_lr={head_lr:.6f}"
            )
            if score < best_gold_score:
                best_gold_score = score
                best_gold = {
                    n:p.detach().cpu().clone()
                    for n,p in compact.named_parameters() if p.requires_grad
                }

    if best_gold:
        with torch.no_grad():
            for n,p in compact.named_parameters():
                if n in best_gold:
                    p.copy_(best_gold[n].to(device=p.device, dtype=p.dtype))

    compact.eval().requires_grad_(False)
    print("V2.5 compact direct-gold dev:", gold_dev_metrics(compact))
    return compact

compact_student = train_compact_model() if RUN_COMPACT_HEAD else None


In [ ]:
#@title 10. REAL held-out test — teacher vs V2.5 full vs V2.5 compact
# The official LocalLLaMA/typed-decisions test split is loaded and used here for the first time.
# It contains 400 cases / 2,000 decisions. Checkpoint selection above used dev only.

test_rows = list(load_dataset("LocalLLaMA/typed-decisions", "all", split="test"))[:TEST_CASES]
test_items = make_items_from_rows(test_rows, SEED + 2, shuffle_items=False)
assert len(test_rows) == TEST_CASES
assert len(test_items) == 5 * len(test_rows), (
    "Official typed-decisions test should contain exactly five decisions per case."
)
print("official test loaded:", len(test_rows), "cases /", len(test_items), "decisions")

@torch.inference_mode()
def collect_test_records(model, items):
    model.eval()
    records = []
    for off in range(0, len(items), BATCH):
        selected = items[off:off+BATCH]
        b = collate_items(selected, int(tokenizer.pad_token_id))
        b = {k:v.to(device) for k,v in b.items()}
        logits, act_logits = _forward(model, b)
        mm = b["marker_mask"].bool()
        probs = torch.softmax(logits.float().masked_fill(~mm, -1e9), dim=-1)

        for j, item in enumerate(selected):
            k = len(item["gold_probs"])
            p = probs[j, :k].detach().cpu().numpy().astype(np.float64)
            p = np.clip(p, 1e-12, None)
            p /= p.sum()
            records.append({
                "id": f"{item['case_id']}/{item['qid']}",
                "case_id": item["case_id"],
                "qid": item["qid"],
                "workflow": item["workflow"],
                "qtype": item["qtype_name"],
                "labels": item["labels"],
                "p": p.tolist(),
                "g": list(map(float, item["gold_probs"])),
                "gold_index": int(item["gold_index"]),
                "pred_index": int(p.argmax()),
                "action_pred": int(act_logits[j].float().argmax().item()),
            })
    return records

def _ece_top_label(records, bins=10):
    total = len(records)
    score = 0.0
    for b in range(bins):
        bucket = []
        for r in records:
            conf = max(r["p"])
            bi = min(bins - 1, int(conf * bins))
            if bi == b:
                bucket.append((conf, r["pred_index"] == r["gold_index"]))
        if bucket:
            avg_conf = float(np.mean([x[0] for x in bucket]))
            avg_acc = float(np.mean([x[1] for x in bucket]))
            score += len(bucket) / total * abs(avg_conf - avg_acc)
    return float(score)

def typed_gold_metrics(records):
    eps = 1e-12
    correct = []
    soft_acc = []
    kl = []
    tv = []
    brier = []
    score_mae = []
    within_one = []

    for r in records:
        p = np.asarray(r["p"], dtype=np.float64)
        g = np.asarray(r["g"], dtype=np.float64)
        pred = int(r["pred_index"])
        gold = int(r["gold_index"])
        correct.append(pred == gold)
        soft_acc.append(float(g[pred]))
        kl.append(float(np.sum(g * np.log(np.clip(g, eps, None) / np.clip(p, eps, None)))))
        tv.append(float(0.5 * np.abs(p - g).sum()))
        brier.append(float(np.square(p - g).sum()))
        if r["qtype"] == "score":
            idx = np.arange(len(p), dtype=np.float64)
            gap = abs(float((idx * p).sum() - (idx * g).sum()))
            score_mae.append(gap)
            within_one.append(gap <= 1.0)

    by_type = {}
    for kind in ("choice", "score", "noul"):
        sub = [r for r in records if r["qtype"] == kind]
        if sub:
            by_type[kind] = float(np.mean([
                r["pred_index"] == r["gold_index"] for r in sub
            ]))

    by_workflow = {}
    for wf in sorted({r["workflow"] for r in records}):
        sub = [r for r in records if r["workflow"] == wf]
        by_workflow[wf] = float(np.mean([
            r["pred_index"] == r["gold_index"] for r in sub
        ]))

    return {
        "decisions": int(len(records)),
        "cases": int(len({r["case_id"] for r in records})),
        "accuracy": float(np.mean(correct)),
        "soft_accuracy": float(np.mean(soft_acc)),
        "kl_from_gold": float(np.mean(kl)),
        "total_variation": float(np.mean(tv)),
        "brier": float(np.mean(brier)),
        "ece_10bin": _ece_top_label(records, bins=10),
        "score_mae": float(np.mean(score_mae)) if score_mae else None,
        "score_within_one": float(np.mean(within_one)) if within_one else None,
        "accuracy_by_type": by_type,
        "accuracy_by_workflow": by_workflow,
    }

def paired_vs_teacher(teacher_records, student_records, draws=3000):
    assert len(teacher_records) == len(student_records)
    t_by_id = {r["id"]: r for r in teacher_records}
    s_by_id = {r["id"]: r for r in student_records}
    assert set(t_by_id) == set(s_by_id)

    ids = sorted(t_by_id)
    t_ok = np.asarray([
        t_by_id[i]["pred_index"] == t_by_id[i]["gold_index"] for i in ids
    ], dtype=np.float64)
    s_ok = np.asarray([
        s_by_id[i]["pred_index"] == s_by_id[i]["gold_index"] for i in ids
    ], dtype=np.float64)

    student_only = int(np.sum((s_ok == 1) & (t_ok == 0)))
    teacher_only = int(np.sum((s_ok == 0) & (t_ok == 1)))
    both_correct = int(np.sum((s_ok == 1) & (t_ok == 1)))
    both_wrong = int(np.sum((s_ok == 0) & (t_ok == 0)))

    agreements, top2_hits, action_agree, js = [], [], [], []
    for i in ids:
        tr, sr = t_by_id[i], s_by_id[i]
        tp = np.asarray(tr["p"], dtype=np.float64)
        sp = np.asarray(sr["p"], dtype=np.float64)
        agreements.append(tr["pred_index"] == sr["pred_index"])
        top2_hits.append(tr["pred_index"] in np.argsort(sp)[-min(2, len(sp)):])
        action_agree.append(tr["action_pred"] == sr["action_pred"])
        mid = 0.5 * (tp + sp)
        j = 0.5 * np.sum(tp * np.log(np.clip(tp, 1e-12, None) / np.clip(mid, 1e-12, None)))
        j += 0.5 * np.sum(sp * np.log(np.clip(sp, 1e-12, None) / np.clip(mid, 1e-12, None)))
        js.append(float(j))

    # Paired bootstrap over CASES (not individual decisions), preserving each case's 5 questions.
    case_to_gaps = {}
    for idx, i in enumerate(ids):
        case_to_gaps.setdefault(t_by_id[i]["case_id"], []).append(float(s_ok[idx] - t_ok[idx]))
    case_gaps = np.asarray([np.mean(v) for v in case_to_gaps.values()], dtype=np.float64)
    rng = np.random.default_rng(SEED + 91)
    boot = np.empty(draws, dtype=np.float64)
    for d in range(draws):
        boot[d] = rng.choice(case_gaps, size=len(case_gaps), replace=True).mean()
    lo, hi = np.percentile(boot, [2.5, 97.5])
    gap = float(s_ok.mean() - t_ok.mean())

    if lo > 0:
        verdict = "student_better"
    elif hi < 0:
        verdict = "teacher_better"
    else:
        verdict = "no_clear_difference"

    return {
        "accuracy_gap_student_minus_teacher": gap,
        "accuracy_gap_95pct_case_bootstrap": [float(lo), float(hi)],
        "paired_verdict": verdict,
        "student_only_correct": student_only,
        "teacher_only_correct": teacher_only,
        "both_correct": both_correct,
        "both_wrong": both_wrong,
        "decision_agreement": float(np.mean(agreements)),
        "teacher_top1_in_student_top2": float(np.mean(top2_hits)),
        "action_head_agreement": float(np.mean(action_agree)),
        "mean_js_divergence": float(np.mean(js)),
        "p95_js_divergence": float(np.percentile(js, 95)),
    }

models = {"teacher": teacher, "v25_full_head": student_full}
if compact_student is not None:
    models["v25_compact"] = compact_student

print("\nRunning untouched official test: 400 cases / 2,000 decisions ...")
test_records = {}
for name, model in models.items():
    print("  evaluating", name)
    test_records[name] = collect_test_records(model, test_items)

# Latency on the same held-out test examples (small fixed probe for repeatability).
test_probe_items = test_items[:min(BATCH, len(test_items))]
test_probe = collate_items(test_probe_items, int(tokenizer.pad_token_id))
test_probe = {k:v.to(device) for k,v in test_probe.items()}

report = {
    "architecture": "v25_window_delta_multiscale_cenn",
    "source_model": SOURCE_MODEL,
    "test_protocol": {
        "dataset": "LocalLLaMA/typed-decisions",
        "config": "all",
        "split": "test",
        "cases": len(test_rows),
        "decisions": len(test_items),
        "used_for_training": False,
        "used_for_checkpoint_selection": False,
        "dev_source": "held-out subset of official train split",
    },
    "full_attention_layers_replaced": full_layers,
    "all_encoder_layers_replaced": replacement_layers,
    "architecture_config": ARCHITECTURE,
    "source_revision": source_dir.name,
    "runtime": {"torch": torch.__version__, "dtype": str(amp_dtype),
                "gpu": torch.cuda.get_device_name(), "transformers": __import__("transformers").__version__},
    "models": {},
}

teacher_gold = typed_gold_metrics(test_records["teacher"])
teacher_perf = benchmark_suite(teacher)
report["models"]["teacher"] = {
    "gold_test": teacher_gold,
    "performance": teacher_perf,
    "quality_vs_teacher": {
        "decision_agreement": 1.0,
        "mean_js_divergence": 0.0,
    },
}

for name, model in models.items():
    if name == "teacher":
        continue
    gold_metrics = typed_gold_metrics(test_records[name])
    perf = benchmark_suite(model)
    paired = paired_vs_teacher(test_records["teacher"], test_records[name])
    report["models"][name] = {
        "gold_test": gold_metrics,
        "performance": perf,
        "speedup_vs_teacher": teacher_perf["median_ms"] / max(perf["median_ms"], 1e-9),
        "parameter_ratio_vs_teacher": perf["total_parameters"] / teacher_perf["total_parameters"],
        "quality_vs_teacher": paired,
    }

# Primary quality ranking: official hard-label accuracy, then KL and Brier as tie-breakers.
ranking = sorted(
    report["models"],
    key=lambda name: (
        -report["models"][name]["gold_test"]["accuracy"],
        report["models"][name]["gold_test"]["kl_from_gold"],
        report["models"][name]["gold_test"]["brier"],
    ),
)
report["real_test_quality_winner"] = ranking[0]
report["real_test_quality_ranking"] = ranking
report["fastest_model"] = min(
    report["models"],
    key=lambda name: report["models"][name]["performance"]["median_ms"],
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
report_path = OUTPUT_DIR / "v25_window_delta_cenn_REAL_TEST_report.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")

print("\n=== REAL HELD-OUT GOLD TEST ===")
print("model               accuracy    KL       Brier    ECE      latency    speed")
print("-" * 82)
for name, result in report["models"].items():
    g = result["gold_test"]
    p = result["performance"]
    speed = 1.0 if name == "teacher" else result["speedup_vs_teacher"]
    print(
        f"{name:18s}  {g['accuracy']:.3%}   {g['kl_from_gold']:.4f}   "
        f"{g['brier']:.4f}   {g['ece_10bin']:.4f}   "
        f"{p['median_ms']:.2f} ms   {speed:.2f}x"
    )
    print("  by type:", {k: f"{v:.2%}" for k, v in g["accuracy_by_type"].items()})

print("\nQUALITY WINNER:", report["real_test_quality_winner"])
print("FASTEST MODEL :", report["fastest_model"])

for name in report["models"]:
    if name == "teacher":
        continue
    q = report["models"][name]["quality_vs_teacher"]
    lo, hi = q["accuracy_gap_95pct_case_bootstrap"]
    print(f"\n{name} vs teacher")
    print(
        f"  accuracy gap : {q['accuracy_gap_student_minus_teacher']:+.2%} "
        f"(95% case-bootstrap CI {lo:+.2%} to {hi:+.2%})"
    )
    print("  verdict      :", q["paired_verdict"])
    print(
        "  student-only / teacher-only correct:",
        q["student_only_correct"], "/", q["teacher_only_correct"],
    )
    print(f"  agreement    : {q['decision_agreement']:.2%}")
    print(f"  top-2 recall : {q['teacher_top1_in_student_top2']:.2%}")
    print(f"  mean JS      : {q['mean_js_divergence']:.5f}")

print("\nsaved:", report_path)


## Interpretation

The teacher is the unchanged source checkpoint. `v25_full_head` is an intermediate with the original Transformer decision head; `v25_compact` is the final pooled-MLP head with no softmax self-attention anywhere. Neither architecture has measured quality until this notebook is run.

Gold accuracy and paired case-bootstrap intervals determine fidelity; teacher agreement is diagnostic. Top-2 recall includes binary questions and is not a standalone quality criterion. Native encoder FFNs and normalization are retained. Windowed Delta is a recurrent/linear-attention-family mechanism, so the final model is a Delta–CeNN hybrid, not a pure CeNN.

Training objectives, LR schedules, step counts and checkpoint-selection formulas match V2.4. Stage A now runs over every encoder layer, increasing total work. The source test split is untouched during this run's training, but has already been used in previous V2.4 research; use a fresh external evaluation for a final publication claim. Source-model training data provenance is not established by this notebook.


In [ ]:
#@title 11. Save, reload and upload ONLY if the final model passes
import os, gc, sys
from huggingface_hub import HfApi, get_token, notebook_login

if compact_student is None:
    print("No final pooled-head model: export skipped. Enable RUN_COMPACT_HEAD and run again.")
else:
    audit = attention_audit(compact_student)
    gate = export_gate(
        report["models"]["teacher"], report["models"]["v25_compact"], audit,
        max_accuracy_drop=MAX_ACCURACY_DROP, min_speedup=MIN_SPEEDUP,
    )
    report["export_gate"] = gate
    report["attention_audit"] = audit
    report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
    print(json.dumps(gate, indent=2))
    if not gate["passed"]:
        print("Model did not pass: report saved, Hugging Face upload skipped.")
    else:
        # Save enough information for exact reconstruction without teacher downloads.
        export_dir = OUTPUT_DIR / "accepted_model"
        save_v25(compact_student, export_dir, tokenizer, ARCHITECTURE, source_cfg, report)
        verification_batch = perf_probes[2]
        compact_student.to(device)
        expected = tuple(x.detach().float().cpu() for x in _forward(compact_student, verification_batch))
        # Offload existing models before loading a second copy on the GPU.
        for existing in models.values():
            existing.to("cpu")
        gc.collect(); torch.cuda.empty_cache()
        reloaded = load_v25(export_dir, device="cuda", backend="fla")
        actual = tuple(x.detach().float().cpu() for x in _forward(reloaded, verification_batch))
        for a, e in zip(actual, expected):
            torch.testing.assert_close(a, e, rtol=0.01, atol=0.01)
        assert attention_audit(reloaded)["passed"]
        report["export_reload_verified"] = True
        (export_dir / "evaluation.json").write_text(json.dumps(report, indent=2, allow_nan=False))
        report_path.write_text(json.dumps(report, indent=2, allow_nan=False))
        del reloaded
        gc.collect(); torch.cuda.empty_cache()
        result = report["models"]["v25_compact"]
        card = f"""---
base_model: {SOURCE_MODEL}
tags:
- tinycenn
- recurrent-memory
- typed-decisions
- custom-code
---
# Laya V2.5: Windowed Delta + Multiscale CeNN

Bidirectional typed-decision encoder. Sliding attention is replaced with overlapping-window Delta;
global attention with multiscale recurrent CeNN; the final head is a pooled-context option MLP.
Encoder FFNs, normalization and residual connections are retained. This is not a causal text generator.

Source checkpoint: {SOURCE_MODEL}, revision `{source_dir.name}`. Follow the source model's license
and usage conditions; this export does not grant a new license to inherited weights.

## Measured evaluation
- Gold test accuracy: {result['gold_test']['accuracy']:.4%}
- Speedup vs unchanged teacher: {result['speedup_vs_teacher']:.3f}x
- Test cases: {result['gold_test']['cases']}
- GPU: {torch.cuda.get_device_name()}
- Dtype: {amp_dtype}
- Full metrics, CI, architecture and acceptance thresholds: `evaluation.json`
- Save/reload equivalence: passed.

Latency covers model forward only on length-stratified dev batches, excluding tokenization.
The 400-case benchmark was used in prior research iterations; this is not independent publication validation.
Action-head outputs are distilled from the teacher, not validated against gold action labels.

## Load
Install `requirements.txt` for CUDA/FLA. The included `tinycenn_lm` folder contains the runtime;
run Python from the downloaded model directory (or add that directory to sys.path).
Use the tokenizer in `tokenizer/` and the typed-decision packing utilities in
`tinycenn_lm.standalone_decision`.

```python
from tinycenn_lm.decision_v25 import load_v25
model = load_v25(".", device="cuda", backend="fla")
# CPU correctness/debug only: load_v25(".", device="cpu", backend="reference")
```
CUDA inference should run under torch.autocast with native BF16 or FP16 as used in evaluation.
The CPU reference recurrence is not a performance backend.
"""
        (export_dir / "README.md").write_text(card)
        print("Accepted checkpoint saved and reload verified:", export_dir)
        if PUSH_TO_HUB:
            token = get_token()
            if not token:
                try:
                    from google.colab import userdata
                    token = userdata.get("HF_TOKEN")
                except Exception:
                    token = None
            if not token:
                notebook_login()
                token = get_token()
            if not token:
                raise RuntimeError("Accepted model saved locally. Authenticate and rerun this cell to upload.")
            api = HfApi(token=token)
            repo_id = HF_REPO_ID.strip() or f"{api.whoami()['name']}/Laya-V25-WindowDelta-CeNN"
            api.create_repo(repo_id, repo_type="model", private=HF_PRIVATE, exist_ok=True)
            commit = api.upload_folder(
                repo_id=repo_id, repo_type="model", folder_path=str(export_dir),
                commit_message="Upload evaluated V2.5 Delta-CeNN decision model",
                ignore_patterns=["**/__pycache__/**", "**/*.pyc"],
            )
            print("Hugging Face model:", f"https://huggingface.co/{repo_id}")
            print("Uploaded commit:", commit.oid)
